# Sommelier Incremental - Stage 1 (Speaker Diarization)

Notebook này được dựng riêng để phát triển và chạy **Stage 1 (Speaker Diarization)** một cách độc lập:
- Clone repo từ github.
- Tự động tải và cấu hình các dependencies **chỉ liên quan đến Stage 1** (nemo-toolkit, pyannote, silero-vad, torch, v.v.).
- Các thư viện không liên quan cho các Stage sau (clearvoice, demucs, panns-inference, chunkformer, whisperx, faster-whisper, openai, tritony) sẽ bị comment lại hoặc không được cài đặt/chạy để tiết kiệm tài nguyên.
- Chạy VAD chunking và diarization sử dụng Sortformer model local.
- Review kết quả Diarization và tạo file ZIP đóng gói kết quả tải về.

## 0. Cấu hình run

Chỉnh các biến bên dưới nếu muốn đổi branch, giới hạn thời lượng test.

In [ ]:
REPO_URL = "https://github.com/lamkdhe180931-arch/sommelier.git"
BRANCH = "codex-stage1-backchannel-recall"

RUN_DIR = "/kaggle/working/run_full"
INPUT_DIR = f"{RUN_DIR}/00_input"
DIAR_DIR = f"{RUN_DIR}/01_diarization"
# MUSIC_DIR = f"{RUN_DIR}/02_music_clean"       # Commented out for Stage 1
# OVERLAP_DIR = f"{RUN_DIR}/03_overlap"          # Commented out for Stage 1
# ASR_DIR = f"{RUN_DIR}/04_asr"                  # Commented out for Stage 1
# EXPORT_DIR = f"{RUN_DIR}/05_export"            # Commented out for Stage 1
# FINAL_DIR = f"{EXPORT_DIR}/final"              # Commented out for Stage 1
# EVAL_DIR = f"{RUN_DIR}/06_eval"                # Commented out for Stage 1
PREVIEW_DIR = f"{RUN_DIR}/preview"
LOG_DIR_PATH = f"{RUN_DIR}/logs"
AUDIO_WAV = f"{INPUT_DIR}/full.wav"

# Để None nếu muốn chạy full audio. Để 300 nếu muốn test nhanh 5 phút.
AUDIO_LIMIT_SECONDS = 300

# Stage 01 backchannel-recall profile: giữ các response ngắn như "dạ", "ừ", "vâng".
SAME_SPEAKER_MERGE_GAP_SECONDS = 0.05
SHORT_BACKCHANNEL_SECONDS = 1.0
SORTFORMER_PAD_ONSET = -0.05
SORTFORMER_PAD_OFFSET = 0.15
SORTFORMER_SOFT_LABEL_THRES = 0.15
SPEAKER_LINK_THRESHOLD = 0.60
SPEAKER_RECLUSTER_THRESHOLD = 0.75

HF_SECRET_NAME = "HF_TOKEN"


In [2]:
from pathlib import Path
import os
import shlex
import subprocess

LOG_DIR = Path(LOG_DIR_PATH)
for _dir in [INPUT_DIR, DIAR_DIR, PREVIEW_DIR, LOG_DIR_PATH]:
    Path(_dir).mkdir(parents=True, exist_ok=True)

def _format_cmd(cmd):
    if isinstance(cmd, (list, tuple)):
        return " ".join(shlex.quote(str(part)) for part in cmd)
    return str(cmd)

def tail_file(path, n=30):
    path = Path(path)
    if not path.exists():
        return ""
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(lines[-n:])

def run_logged(cmd, log_name, cwd=None, env=None, shell=False, tail=20):
    log_path = LOG_DIR / log_name
    cwd = cwd or os.getcwd()
    print("Running:", _format_cmd(cmd))
    print("Log:", log_path)
    with open(log_path, "w", encoding="utf-8", errors="replace") as log:
        proc = subprocess.run(
            cmd,
            cwd=cwd,
            env=env,
            shell=shell,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )
    print("Exit code:", proc.returncode)
    if tail:
        log_tail = tail_file(log_path, n=tail)
        if log_tail:
            print(f"--- last {tail} log lines ---")
            print(log_tail)
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return log_path

def export_audio_preview(audio_segment, out_path, seconds=30):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    audio_segment[: int(seconds * 1000)].export(out_path, format="wav")
    return out_path

## 1. Clone repo

In [3]:
import os
import shutil
import subprocess
from pathlib import Path

os.chdir("/kaggle/working")
repo_dir = Path("/kaggle/working/sommelier")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

run_logged(["git", "clone", "-b", BRANCH, REPO_URL, str(repo_dir)], "01_clone_repo.log", cwd="/kaggle/working", tail=20)
os.chdir(repo_dir / "podcast-pipeline")
print("cwd:", os.getcwd())
print("branch:", subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip())
print("commit:", subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())

Running: git clone -b test-divide-stage https://github.com/lamkdhe180931-arch/sommelier.git /kaggle/working/sommelier
Log: /kaggle/working/run_full/logs/01_clone_repo.log
Exit code: 0
--- last 20 log lines ---
Cloning into '/kaggle/working/sommelier'...
cwd: /kaggle/working/sommelier/podcast-pipeline
branch: test-divide-stage
commit: 00829b1 Add speaker re-clustering functionality and enhance run viewer


## 2. Cài dependencies từ internet

Tải các gói cài đặt tối thiểu cho Stage 1. Lọc và comment các thư viện không dùng ở Stage 1.

In [4]:
# import os
# from pathlib import Path

# os.chdir("/kaggle/working/sommelier/podcast-pipeline")

# run_logged(["apt-get", "update", "-y"], "02_apt_update.log", tail=10)
# run_logged(["apt-get", "install", "-y", "ffmpeg", "git", "git-lfs"], "03_apt_install.log", tail=10)
# run_logged(["python", "-m", "pip", "install", "-U", "pip", "setuptools", "wheel", "packaging", "ninja"], "04_pip_base.log", tail=12)

# req = Path("requirements.txt").read_text(encoding="utf-8")
# filtered = [line for line in req.splitlines() if "nemo-toolkit[all]" not in line]
# Path("requirements-kaggle.txt").write_text("\n".join(filtered) + "\n", encoding="utf-8")
# run_logged(["python", "-m", "pip", "install", "-r", "requirements-kaggle.txt"], "05_pip_requirements.log", tail=20)

# run_logged(["python", "-m", "pip", "uninstall", "-y", "nemo-toolkit", "lightning", "pytorch-lightning"], "06_pip_uninstall_nemo.log", tail=8)
# run_logged(["python", "-m", "pip", "install", "lightning==2.4.0", "pytorch-lightning==2.5.2"], "07_pip_lightning.log", tail=12)
# run_logged(["python", "-m", "pip", "install", "nemo-toolkit[asr]==2.4.0"], "08_pip_nemo_asr.log", tail=20)

# run_logged(["python", "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"], "09_pip_uninstall_torch.log", tail=8)
# run_logged([
#     "python", "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
#     "torch==2.7.1", "torchaudio==2.7.1", "torchvision==0.22.1",
#     "--index-url", "https://download.pytorch.org/whl/cu126",
# ], "10_pip_torch_stack.log", tail=20)

# run_logged(["python", "-m", "pip", "install", "pillow<12.0"], "11_pip_pillow.log", tail=8)
# run_logged(["python", "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "--no-deps", "torchmetrics==1.7.4"], "12_pip_torchmetrics.log", tail=8)
# run_logged([
#     "python", "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
#     "numpy==2.2.6", "numba==0.61.2", "llvmlite==0.44.0",
# ], "13_pip_numpy_numba.log", tail=12)

# run_logged(["python", "-m", "pip", "install", "--no-cache-dir", "clearvoice==0.1.2"], "14_pip_clearvoice.log", tail=20)

# print("Dependency install logs saved in:", LOG_DIR)


## 3. Kiểm tra môi trường

Kiểm tra môi trường GPU, CUDA và các import liên quan tới Stage 1.

In [5]:
import importlib.metadata as importlib_metadata
import shutil
import subprocess
import numpy, numba, torch

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not found; kiểm tra torch.cuda bên dưới.")

print("nemo-toolkit:", importlib_metadata.version("nemo-toolkit"))
# print("chunkformer:", importlib_metadata.version("chunkformer")) # Commented out for Stage 1
print("numpy:", numpy.__version__)
print("numba:", numba.__version__)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print("GPU", i, torch.cuda.get_device_name(i))

# whisperx is commented out as it is only needed for Stage 04 ASR
# import whisperx
# print("whisperx ok")

import nemo.collections.asr as nemo_asr
print("nemo asr ok")

from nemo.collections.asr.models import SortformerEncLabelModel
print("sortformer import ok")

Sun May 31 15:35:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

[NeMo W 2026-05-31 15:36:06 nemo_logging:405] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
      m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
    
[NeMo W 2026-05-31 15:36:06 nemo_logging:405] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
      m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
    
[NeMo W 2026-05-31 15:36:06 nemo_logging:405] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(flt)p?( \(default\))?$', token):
    
[NeMo W 2026-05-31 15:36:06 nemo_logging:405] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(dbl)p?( \(default\))?$', token):
    


nemo asr ok
sortformer import ok


## 4. Gắn Hugging Face token vào config

Cần thiết để tải model gated như `pyannote/embedding` phục vụ speaker linking.

In [6]:
import json
from kaggle_secrets import UserSecretsClient
from huggingface_hub import whoami

token = UserSecretsClient().get_secret(HF_SECRET_NAME)
print("HF token:", token[:8] + "..." if token else "missing")
print(whoami(token=token))

with open("config.json", "r", encoding="utf-8") as f:
    cfg = json.load(f)

cfg["huggingface_token"] = token

with open("config.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)

print("config.json updated")

HF token: hf_yFiyW...
{'type': 'user', 'id': '69e9fc5c89e67b7de7f4737a', 'name': 'lamkieu2282', 'fullname': 'kieulam', 'email': 'kdlam1522004@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1780272000, 'isPro': False, 'avatarUrl': '/avatars/b022b37527a9464d3bc5dd436a1de8cf.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'newtoken', 'role': 'read', 'createdAt': '2026-05-28T09:22:37.382Z'}}}
config.json updated


## 5. Tìm audio input và chuẩn hóa audio

Chuẩn hóa âm thanh về định dạng mono 16 kHz để tương thích tối đa với Silero VAD và Sortformer.

In [7]:
from pathlib import Path

audio_exts = {".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg"}
audio_candidates = sorted(
    p for p in Path("/kaggle/input").rglob("*")
    if p.is_file() and p.suffix.lower() in audio_exts
)

if not audio_candidates:
    raise FileNotFoundError("Không tìm thấy audio trong /kaggle/input. Hãy Add Input hoặc Upload audio trước.")

AUDIO_IN = str(audio_candidates[0])
print("AUDIO_IN:", AUDIO_IN)
print("AUDIO_WAV:", AUDIO_WAV)
print("RUN_DIR:", RUN_DIR)

Path(INPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

cmd = ["ffmpeg", "-hide_banner", "-y", "-i", AUDIO_IN]
if AUDIO_LIMIT_SECONDS:
    cmd += ["-t", str(AUDIO_LIMIT_SECONDS)]
cmd += ["-ac", "1", "-ar", "16000", AUDIO_WAV]
run_logged(cmd, "00_prepare_audio_ffmpeg.log", cwd="/kaggle/working", tail=15)

AUDIO_IN: /kaggle/input/datasets/thuhinl123/data-input/MP3Now.com_YouTube_Di-muon-noi-cung-chang-bang-chuyen-di-cu_Media_Zho159IWRWo_009_128k.mp3
AUDIO_WAV: /kaggle/working/run_full/00_input/full.wav
RUN_DIR: /kaggle/working/run_full
Running: ffmpeg -hide_banner -y -i /kaggle/input/datasets/thuhinl123/data-input/MP3Now.com_YouTube_Di-muon-noi-cung-chang-bang-chuyen-di-cu_Media_Zho159IWRWo_009_128k.mp3 -t 300 -ac 1 -ar 16000 /kaggle/working/run_full/00_input/full.wav
Log: /kaggle/working/run_full/logs/00_prepare_audio_ffmpeg.log
Exit code: 0
--- last 15 log lines ---
  Stream #0:0 -> #0:0 (mp3 (mp3float) -> pcm_s16le (native))
Press [q] to stop, [?] for help
Output #0, wav, to '/kaggle/working/run_full/00_input/full.wav':
  Metadata:
    major_brand     : dash
    minor_version   : 0
    compatible_brands: iso6mp41
    ISFT            : Lavf58.76.100
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 16000 Hz, mono, s16, 256 kb/s
    Metadata:
      encoder         : Lavc58.134.10

PosixPath('/kaggle/working/run_full/logs/00_prepare_audio_ffmpeg.log')

In [8]:
from pathlib import Path
from pydub import AudioSegment
from IPython.display import Audio, display

audio = AudioSegment.from_file(AUDIO_WAV)
print("Audio:", AUDIO_WAV)
print("Duration seconds:", len(audio) / 1000)
print("Frame rate:", audio.frame_rate)
print("Channels:", audio.channels)

preview_path = export_audio_preview(audio, Path(INPUT_DIR) / "preview_input_30s.wav", seconds=30)
print("Preview first 30s:", preview_path)
display(Audio(str(preview_path)))

Audio: /kaggle/working/run_full/00_input/full.wav
Duration seconds: 300.0
Frame rate: 16000
Channels: 1
Preview first 30s: /kaggle/working/run_full/00_input/preview_input_30s.wav


## 6. Trace VAD chunking

Chia âm thanh lớn thành các chunk nhỏ dựa trên khoảng lặng (VAD) để hỗ trợ bộ giải mã Sortformer chạy mượt mà.

In [9]:
import os
import json
import shutil
from pathlib import Path
import pandas as pd
from pydub import AudioSegment
from IPython.display import display

os.chdir("/kaggle/working/sommelier/podcast-pipeline")

import stage_common
import main_original_ASR_MoE as pipeline

cfg = pipeline.load_cfg("config.json")
logger = pipeline.Logger.get_logger()
pipeline.cfg = cfg
pipeline.logger = logger

device_name = "cuda" if pipeline.torch.cuda.is_available() else "cpu"
device = pipeline.torch.device(device_name)
pipeline.device_name = device_name
pipeline.device = device
pipeline.vad = pipeline.silero_vad.SileroVAD(device=device)

sample_rate = int(cfg["entrypoint"]["SAMPLE_RATE"])
audio_info = stage_common.load_audio_info(AUDIO_WAV, sample_rate)
diar_chunks, temp_chunk_dir = pipeline.prepare_diarization_chunks(AUDIO_WAV, audio_info)

chunk_dir = Path(DIAR_DIR) / "vad_chunks"
chunk_dir.mkdir(parents=True, exist_ok=True)

trace_chunks = []
for idx, chunk in enumerate(diar_chunks):
    src = Path(chunk["path"])
    dst = chunk_dir / f"chunk_{idx:03d}.wav"
    shutil.copy2(src, dst)
    duration = AudioSegment.from_file(dst).duration_seconds
    trace_chunks.append({
        "index": f"{idx:03d}",
        "path": str(dst),
        "offset": float(chunk["offset"]),
        "duration": float(duration),
        "start": float(chunk["offset"]),
        "end": float(chunk["offset"] + duration),
    })

if temp_chunk_dir:
    shutil.rmtree(temp_chunk_dir, ignore_errors=True)

stage_common.dump_json({
    "audio_path": AUDIO_WAV,
    "sample_rate": sample_rate,
    "chunks": trace_chunks,
    "metadata": {"stage": "vad_chunk_trace"}
}, Path(DIAR_DIR) / "trace_vad_chunks.json")

df_chunks = pd.DataFrame(trace_chunks)
print("VAD chunks:", len(df_chunks))
display(df_chunks.head(20))

2026-05-31 15:36:13.009610: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780241773.179109    1189 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780241773.226679    1189 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780241773.623508    1189 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780241773.623535    1189 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780241773.623537    1189 computation_placer.cc:177] computation placer alr

Initialize logger for main
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip


2026-05-31 15:36:45,718 - main - INFO - Pre-diarization chunking created 2 chunks (max 180s) from full.wav
INFO:main:Pre-diarization chunking created 2 chunks (max 180s) from full.wav


VAD chunks: 2


,index,path,offset,duration,start,end
0,000,/kaggle/working/run_full/01_diarization/vad_ch...,0.000,106.816,0.000,106.816
1,001,/kaggle/working/run_full/01_diarization/vad_ch...,106.816,193.184,106.816,300.000


In [10]:
from pathlib import Path
from pydub import AudioSegment
from IPython.display import Audio, display

if trace_chunks:
    first = trace_chunks[0]
    print(first)
    chunk_audio = AudioSegment.from_file(first["path"])
    preview_path = export_audio_preview(chunk_audio, Path(DIAR_DIR) / "preview_vad_chunk_0_30s.wav", seconds=30)
    print("Preview first 30s of chunk 0:", preview_path)
    display(Audio(str(preview_path)))

{'index': '000', 'path': '/kaggle/working/run_full/01_diarization/vad_chunks/chunk_000.wav', 'offset': 0.0, 'duration': 106.816, 'start': 0.0, 'end': 106.816}
Preview first 30s of chunk 0: /kaggle/working/run_full/01_diarization/preview_vad_chunk_0_30s.wav


## 7. Stage 01 - Speaker diarization

Chạy model Sortformer local để nhận dạng người nói, căn chỉnh speaker ID qua các chunk bằng `pyannote/embedding`.

In [ ]:
# Stage 1 now applies Sortformer sensitivity tuning inside stage_01_diarize.py.
# The old in-notebook monkey patch did not affect the subprocess launched below.
print("Sortformer backchannel tuning will be applied by stage_01_diarize.py via --sortformer-soft-label-thres.")


In [ ]:
import os
os.chdir("/kaggle/working/sommelier/podcast-pipeline")

run_logged([
    "python", "stage_01_diarize.py",
    "--input_audio", AUDIO_WAV,
    "--seg_th", "0.02",
    "--min_cluster_size", "3",
    "--clust_th", "0.35",
    "--out", f"{DIAR_DIR}/diarization.json",
    "--merge_gap", str(SAME_SPEAKER_MERGE_GAP_SECONDS),
    "--max_segment_duration", "30.0",
    "--same_speaker_merge_gap", str(SAME_SPEAKER_MERGE_GAP_SECONDS),
    "--short_backchannel_seconds", str(SHORT_BACKCHANNEL_SECONDS),
    "--sortformer-pad-onset", str(SORTFORMER_PAD_ONSET),
    "--sortformer-pad-offset", str(SORTFORMER_PAD_OFFSET),
    "--sortformer-soft-label-thres", str(SORTFORMER_SOFT_LABEL_THRES),
    "--speaker-link-threshold", str(SPEAKER_LINK_THRESHOLD),
    "--speaker-recluster-threshold", str(SPEAKER_RECLUSTER_THRESHOLD),
], "18_stage_01_diarize.log", tail=35)


## 8. Review kết quả Diarization

In [13]:
import json
import pandas as pd
from IPython.display import display

with open(f"{DIAR_DIR}/diarization.json", "r", encoding="utf-8") as f:
    diar = json.load(f)

diar_segments = diar["segments"]
df_diar = pd.DataFrame(diar_segments)
df_diar["dur"] = df_diar["end"].astype(float) - df_diar["start"].astype(float)

print("File:", f"{DIAR_DIR}/diarization.json")
print("Total segments:", len(df_diar))
print("Speakers:", sorted(df_diar["speaker"].unique()) if len(df_diar) else [])
print("Duration median:", df_diar["dur"].median() if len(df_diar) else 0)
print("Duration mean:", df_diar["dur"].mean() if len(df_diar) else 0)
print("< 1s:", int((df_diar["dur"] < 1).sum()) if len(df_diar) else 0)
print("< 2s:", int((df_diar["dur"] < 2).sum()) if len(df_diar) else 0)
print("< 3s:", int((df_diar["dur"] < 3).sum()) if len(df_diar) else 0)

display(df_diar[["index", "start", "end", "dur", "speaker"]].head(30))

File: /kaggle/working/run_full/01_diarization/diarization.json
Total segments: 92
Speakers: ['SPEAKER_00', 'SPEAKER_01', 'SPEAKER_02', 'SPEAKER_03', 'SPEAKER_04', 'SPEAKER_05']
Duration median: 1.1099999999999852
Duration mean: 3.131739130434773
< 1s: 45
< 2s: 55
< 3s: 62


,index,start,end,dur,speaker
0,00000,0.130,2.000,1.87,SPEAKER_00
1,00001,3.090,17.120,14.03,SPEAKER_01
2,00002,17.970,18.400,0.43,SPEAKER_01
3,00003,19.250,25.040,5.79,SPEAKER_00
4,00004,21.010,26.800,5.79,SPEAKER_01
5,00005,25.890,26.800,0.91,SPEAKER_00
6,00006,27.410,38.960,11.55,SPEAKER_01
7,00007,40.450,43.600,3.15,SPEAKER_00
8,00008,44.930,49.920,4.99,SPEAKER_00
9,00009,48.130,48.160,0.03,SPEAKER_01


In [14]:
from pathlib import Path
import json
import pandas as pd
from pydub import AudioSegment
from IPython.display import Audio, display

def tail_file(path, n=30):
    path = Path(path)
    if not path.exists():
        return ""
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(lines[-n:])

def show_stage_log(log_path, lines=25):
    path = Path(log_path)
    print("\n" + "=" * 90)
    print(path)
    print("=" * 90)
    if path.exists():
        print(tail_file(path, lines))
    else:
        print("Log chưa tồn tại. Hãy chạy stage tương ứng trước.")

print("Stage 01 log:")
show_stage_log(Path(LOG_DIR_PATH) / "18_stage_01_diarize.log", lines=25)

with open(f"{DIAR_DIR}/diarization.json", "r", encoding="utf-8") as f:
    diar = json.load(f)
diar_segments = diar["segments"]
full_audio = AudioSegment.from_file(AUDIO_WAV)

def listen_diarization_segment(i, pad=0.15, max_seconds=30):
    if i >= len(diar_segments):
        print(f"Segment index {i} out of bounds.")
        return
    s = diar_segments[i]
    start = max(0.0, float(s["start"]) - pad)
    end = float(s["end"]) + pad
    if end - start > max_seconds:
        end = start + max_seconds
    
    speaker = s.get("speaker", "UNKNOWN")
    print(f"Segment {i} | Speaker: {speaker} | Time: {s['start']}s - {s['end']}s")
    
    tmp_clip = Path(PREVIEW_DIR) / f"preview_diar_{i:04d}.wav"
    tmp_clip.parent.mkdir(parents=True, exist_ok=True)
    full_audio[int(start * 1000):int(end * 1000)].export(tmp_clip, format="wav")
    display(Audio(str(tmp_clip)))

print("\nCall listen_diarization_segment(i) để nghe thử segment mong muốn, ví dụ: listen_diarization_segment(0)")
if diar_segments:
    listen_diarization_segment(0)

Stage 01 log:

/kaggle/working/run_full/logs/18_stage_01_diarize.log
    num_workers: 18
    validation_mode: true
    use_lhotse: false
    use_bucketing: false
    drop_last: false
    pin_memory: true
    window_stride: 0.01
    subsampling_factor: 8
    
Initialize logger for main
[NeMo I 2026-05-31 15:37:09 nemo_logging:393] PADDING: 16
[NeMo I 2026-05-31 15:37:10 nemo_logging:393] Model SortformerEncLabelModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--diar_sortformer_4spk-v1/snapshots/9f17b10df44c0a4c8f3c86fbddc9ee2d6ab9ac08/diar_sortformer_4spk-v1.nemo.
2026-05-31 15:37:16,102 - main - INFO - Pre-diarization chunking created 2 chunks (max 180s) from full.wav
INFO:main:Pre-diarization chunking created 2 chunks (max 180s) from full.wav
[NeMo I 2026-05-31 15:37:16 nemo_logging:393] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.

Diarizing: 100%|█████████████████████████████████| 1/1 [00:01<00:0

## 9. Đóng gói kết quả Diarization

Nén thư mục kết quả Diarization (bao gồm logs và json) thành một tệp tin ZIP để tiện tải về máy local.

In [15]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

diar_dir = Path(DIAR_DIR)
zip_base = Path("/kaggle/working/diarization_results")

zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(diar_dir.parent),
    base_dir=diar_dir.name,
)

print("Created:", zip_path)
display(FileLink(zip_path))

Created: /kaggle/working/diarization_results.zip


/kaggle/working/diarization_results.zip

In [16]:
import json
import pandas as pd

# Đường dẫn đến file diarization.json kết quả của Stage 1
diar_json_path = "/kaggle/working/run_full/01_diarization/diarization.json"

try:
    with open(diar_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # 1. Hiển thị thông tin tổng quan (Metadata)
    print("=" * 60)
    print("THÔNG TIN TỔNG QUAN (METADATA):")
    print("=" * 60)
    metadata = data.get("metadata", {})
    for key, val in metadata.items():
        if key != "postprocess" and key != "speaker_recluster":
            print(f"- {key}: {val}")
    
    # 2. Hiển thị TOÀN BỘ các phân đoạn người nói (Segments) dưới dạng bảng
    print("\n" + "=" * 60)
    print("TOÀN BỘ PHÂN ĐOẠN NGƯỜI NÓI (SEGMENTS):")
    print("=" * 60)
    segments = data.get("segments", [])
    df = pd.DataFrame(segments)
    
    # Tính thêm cột độ dài phân đoạn (Duration) để tiện theo dõi
    if not df.empty:
        df["duration"] = df["end"].astype(float) - df["start"].astype(float)
        df["duration"] = df["duration"].round(2)
        
        # Cấu hình Pandas hiển thị tối đa toàn bộ dòng và cột, không bị ẩn dấu ba chấm (...)
        pd.set_option("display.max_rows", None)
        pd.set_option("display.max_columns", None)
        pd.set_option("display.width", 1000)
        
        # Hiển thị bảng dữ liệu hoàn chỉnh
        display(df[["index", "start", "end", "duration", "speaker"]])
    else:
        print("Không có phân đoạn nào được tìm thấy.")
        
except FileNotFoundError:
    print(f"LỖI: Không tìm thấy file tại đường dẫn: {diar_json_path}")
    print("Vui lòng đảm bảo bạn đã chạy thành công Stage 1 trước đó.")
except Exception as e:
    print(f"Đã xảy ra lỗi khi đọc file: {e}")

THÔNG TIN TỔNG QUAN (METADATA):
- stage: diarize
- processing_time_seconds: 7.718713760375977
- rt_factor: 0.025729045867919922
- seg_th: 0.02
- min_cluster_size: 3
- clust_th: 0.35
- speaker_link_threshold: 0.9
- same_speaker_merge_gap_seconds: 0.05
- short_backchannel_seconds: 0.3
- speaker_recluster_threshold: 0.0

TOÀN BỘ PHÂN ĐOẠN NGƯỜI NÓI (SEGMENTS):


,index,start,end,duration,speaker
0,00000,0.130,2.000,1.87,SPEAKER_00
1,00001,3.090,17.120,14.03,SPEAKER_01
2,00002,17.970,18.400,0.43,SPEAKER_01
3,00003,19.250,25.040,5.79,SPEAKER_00
4,00004,21.010,26.800,5.79,SPEAKER_01
5,00005,25.890,26.800,0.91,SPEAKER_00
6,00006,27.410,38.960,11.55,SPEAKER_01
7,00007,40.450,43.600,3.15,SPEAKER_00
8,00008,44.930,49.920,4.99,SPEAKER_00
9,00009,48.130,48.160,0.03,SPEAKER_01


In [17]:
import json
from pathlib import Path
from pydub import AudioSegment
from IPython.display import Audio, display, HTML

# Đường dẫn dữ liệu
diar_json_path = "/kaggle/working/run_full/01_diarization/diarization.json"
audio_wav_path = "/kaggle/working/run_full/00_input/full.wav"
preview_dir = Path("/kaggle/working/run_full/preview/all_segments")
preview_dir.mkdir(parents=True, exist_ok=True)

try:
    # 1. Đọc dữ liệu phân đoạn
    with open(diar_json_path, "r", encoding="utf-8") as f:
        diar = json.load(f)
    segments = diar.get("segments", [])
    
    # Load file âm thanh gốc vào bộ nhớ
    print("Đang tải file âm thanh gốc (vui lòng chờ vài giây)...")
    full_audio = AudioSegment.from_file(audio_wav_path)
    total_segs = len(segments)
    print(f"Đã tải xong! Tổng số phân đoạn tìm thấy: {total_segs}")
    print("=" * 70)

    # =========================================================================
    # CẤU HÌNH PHẠM VI NGHE Ở ĐÂY:
    # =========================================================================
    START_INDEX = 0    # Phân đoạn bắt đầu nghe
    END_INDEX = 10     # Phân đoạn kết thúc (Bạn có thể nâng lên tùy ý, ví dụ: 20, 30)
    PADDING = 0.15      # Khoảng đệm âm thanh trước/sau segment (giây) để tránh mất chữ
    # =========================================================================

    # Giới hạn phạm vi hợp lệ
    end_idx = min(END_INDEX, total_segs)
    start_idx = max(0, START_INDEX)

    print(f"Đang trích xuất và hiển thị âm thanh từ đoạn {start_idx} đến {end_idx - 1}:")
    print("=" * 70)

    for i in range(start_idx, end_idx):
        s = segments[i]
        start_time = float(s["start"])
        end_time = float(s["end"])
        duration = round(end_time - start_time, 2)
        speaker = s.get("speaker", "UNKNOWN")
        
        # Cắt âm thanh kèm padding
        pad_start = max(0.0, start_time - PADDING)
        pad_end = end_time + PADDING
        
        # Xuất file wav tạm
        out_name = f"seg_{i:04d}_{speaker}.wav"
        out_path = preview_dir / out_name
        
        # Trích xuất và xuất file
        full_audio[int(pad_start * 1000):int(pad_end * 1000)].export(out_path, format="wav")
        
        # Hiển thị trình phát nhạc kèm thông tin đẹp mắt bằng HTML
        display(HTML(f"""
            <div style="border-left: 5px solid #00c0ef; padding-left: 15px; margin-bottom: 15px; background-color: #f4f6f9; padding-top: 5px; padding-bottom: 5px; border-radius: 4px;">
                <span style="font-weight: bold; color: #3c8dbc;">Segment {i} ({s.get('index', '')})</span> | 
                <span style="font-weight: bold; color: #ff851b;">{speaker}</span> | 
                Thời gian: <b>{start_time:.2f}s - {end_time:.2f}s</b> (Độ dài: <b>{duration}s</b>)
            </div>
        """))
        display(Audio(str(out_path)))
        
except FileNotFoundError:
    print("LỖI: Chưa có file diarization.json hoặc full.wav. Hãy chạy thành công Stage 1 trước.")
except Exception as e:
    print(f"Đã xảy ra lỗi: {e}")

Đang tải file âm thanh gốc (vui lòng chờ vài giây)...
Đã tải xong! Tổng số phân đoạn tìm thấy: 92
Đang trích xuất và hiển thị âm thanh từ đoạn 0 đến 9:
